# DyslexAI — Notebook 03: Preprocessing, Dataset & DataLoader

**Final Year Project — Software Engineering**
**Phase 2B — Hands-on implementation | Handwriting modality**

---

## What this notebook does

Notebook 01 audited the data. Notebook 02 decided what to keep and how to split it.
This notebook builds the **plumbing that feeds images into a neural network**.

Still no training. By the end we will have verified that a batch of correctly preprocessed
image tensors comes out of a `DataLoader` with the shape a model expects — and we will have
*looked* at those tensors to confirm they still contain handwriting rather than noise.

That last check matters more than it sounds. Most silent failures in image pipelines are
invisible in the numbers and obvious the moment you plot the tensor.

---

## Stage map

| Stage | Name |
|---|---|
| 0 | Setup and load the split manifests |
| 1 | Preprocessing decisions — size, aspect ratio, padding |
| 2 | Channels — what RGB vs grayscale means for us |
| 3 | Normalisation |
| 4 | The transform pipelines |
| 5 | The PyTorch `Dataset` class |
| 6 | The `DataLoader` |
| 7 | Debugging workflow — shapes, one batch, visual confirmation |
| 8 | Class weights (computed, not yet applied) |
| 9 | Save the preprocessing config |

---

## Design principle for this notebook

Every experiment must read from a **split manifest CSV**, never from folders.

Folder-based loading (`ImageFolder`) would silently undo all of Notebook 02's work — it would
re-include the duplicates and contradictions we removed, because it just reads whatever is on
disk. Reading from a manifest means the cleaning decisions are enforced at load time and are
visible in a file anyone can inspect.

---
# STAGE 0 — Setup

### 0.1 Mount Drive and check the GPU

From this notebook onward a GPU genuinely helps. Set
`Runtime → Change runtime type → T4 GPU` before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json, random, subprocess, time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torchvision

print(f'torch       : {torch.__version__}')
print(f'torchvision : {torchvision.__version__}')
print(f'CUDA        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
else:
    print('GPU         : none — everything still works, just slower')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device      : {DEVICE}')

In [ ]:
# ---- Paths -----------------------------------------------------------------
DRIVE_ROOT   = Path('/content/drive/MyDrive')
PROJECT_DIR  = DRIVE_ROOT / 'DyslexAI'
ARCHIVE_DIR  = PROJECT_DIR / 'archives'
RAW_DIR      = Path('/content/raw')

INTERIM_DIR  = PROJECT_DIR / 'data' / 'interim'
CLEANED_DIR  = PROJECT_DIR / 'data' / 'cleaned'
SPLITS_DIR   = PROJECT_DIR / 'data' / 'splits'
REPORTS_DIR  = PROJECT_DIR / 'reports'
FIGURES_DIR  = PROJECT_DIR / 'results' / 'figures'
EXPERIMENTS_DIR = PROJECT_DIR / 'experiments'

for d in [RAW_DIR, FIGURES_DIR, EXPERIMENTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---- Reproducibility -------------------------------------------------------
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print('Paths ready. Seed =', SEED)

In [ ]:
# ---- Re-extract images (Colab wipes local disk between sessions) -----------
!apt-get -qq install -y libarchive-tools > /dev/null 2>&1

DEST = RAW_DIR / 'dataset_a'
if not DEST.exists() or not any(DEST.rglob('*.jpeg')):
    DEST.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    subprocess.run(['bsdtar', '-xf',
                    str(ARCHIVE_DIR / 'Complete_And_Balance_Hand-written_Dataset.rar'),
                    '-C', str(DEST)], check=False)
    print(f'Extracted in {time.time()-t0:.1f}s')
else:
    print('Already extracted this session.')


def find_class_root(search_root: Path, expected=('Yes', 'No')) -> Path:
    want = {c.lower() for c in expected}
    for d in search_root.rglob('*'):
        if d.is_dir() and want.issubset({c.name.lower() for c in d.iterdir() if c.is_dir()}):
            return d
    raise FileNotFoundError(f'Class folders {expected} not found under {search_root}')


DATASET_A_ROOT = find_class_root(DEST)
print('Class root:', DATASET_A_ROOT)

### 0.2 Load the split manifests

Three manifests from Notebook 02. We attach the absolute path at load time, so the manifests
stay portable.

In [ ]:
def load_split(name: str) -> pd.DataFrame:
    """Load a split manifest and attach absolute paths for this session."""
    df = pd.read_csv(SPLITS_DIR / f'{name}_split.csv')
    df['abs_path'] = df['relative_path'].map(lambda rp: str(DATASET_A_ROOT / rp))

    missing = [p for p in df['abs_path'] if not Path(p).exists()]
    assert not missing, f'{len(missing)} files missing from {name}'
    return df


splits = {name: load_split(name) for name in ['S1_raw', 'S1_clean', 'S2_grouped']}

for name, df in splits.items():
    counts = df['split'].value_counts()
    print(f'{name:<12} n={len(df):>4}  '
          f'train={counts.get("train", 0):>4}  '
          f'val={counts.get("val", 0):>3}  '
          f'test={counts.get("test", 0):>3}')

---
# STAGE 1 — Preprocessing Decisions

### The problem we measured

Notebook 01 found **538 distinct pixel sizes** across 852 images, widths from 119 to 1232 px,
and aspect ratios from **0.25 to 2.17**.

A neural network needs every input tensor to be the same shape, so we must resize. The
question is *how*, and for handwriting the answer is not obvious.

### Three ways to make images the same size

**Option A — resize straight to a square.**
Fast and simple, but it **stretches the image**. A character in a 1232×500 photo gets squashed
horizontally; the same character in a 500×1232 photo gets stretched. The network then sees two
different shapes for the same letter. For handwriting — where *shape is the entire signal* —
this destroys information we care about.

**Option B — centre crop to a square, then resize.**
Preserves shape but **cuts pixels off the edges**. Our images are tight crops of single
characters, so cropping risks slicing off the very stroke that matters (an Urdu dot, the tail
of a ژ).

**Option C — pad to a square, then resize.** ← *our choice*
Add blank margin to the shorter side until the image is square, then resize. Nothing is
stretched and nothing is cut. The cost is some wasted blank area on extreme aspect ratios.

### What colour to pad with

Padding with black would put a hard dark border around light paper — a strong artificial edge
the network can learn from, and worse, the *amount* of black would correlate with the original
aspect ratio. If aspect ratio happens to differ between classes, the model could classify on
border thickness alone.

We pad with a value **measured from each image's own corners**, which on these tight character
crops are almost always blank page.

Two things this avoids. Guessing "white" would be wrong — these are phone photographs under
classroom lighting, not scans, and the paper averages around 166 rather than 255. And a single
global fill would still show as a visible band, because every photograph has its own exposure.
If brightness happens to differ between classes or capture sessions, that band becomes a cue the
network can learn from instead of the handwriting.

### Output size

We use **224×224** throughout.

It is the native input size for ImageNet-pretrained ResNet and EfficientNet, so the simple CNN
and the transfer-learning models all see identically preprocessed data. That makes the
comparison between them fair — any difference in results comes from the model, not from the
preprocessing.

In [ ]:
# ---- Preprocessing constants ----------------------------------------------
IMG_SIZE = 224          # final square side, matches ImageNet-pretrained models

# The padding colour must match the paper, or the border becomes a visible edge
# the network can learn from. Rather than guessing, we MEASURE it: the four
# corners of these crops are almost always blank page.
from PIL import Image

sample_paths = list(splits['S1_clean']['abs_path'].head(50))
corner_values = []
for p in sample_paths:
    arr = np.asarray(Image.open(p).convert('RGB'))
    corners = np.concatenate([arr[:8, :8].reshape(-1, 3),  arr[:8, -8:].reshape(-1, 3),
                              arr[-8:, :8].reshape(-1, 3), arr[-8:, -8:].reshape(-1, 3)])
    corner_values.append(corners.mean(axis=0))

corner_mean = np.mean(corner_values, axis=0)
PAD_VALUE = int(round(corner_mean.mean()))

print(f'measured paper colour over {len(sample_paths)} images: '
      f'R={corner_mean[0]:.0f} G={corner_mean[1]:.0f} B={corner_mean[2]:.0f}')
print(f'PAD_VALUE = {PAD_VALUE}')
print()
print('Note how much darker this is than plain white (255). These are phone')
print('photographs under classroom lighting, not scans. Padding with white')
print('would have put a bright band around every non-square image.')

---
# STAGE 2 — Channels: RGB vs Grayscale

### What a channel is

A digital image is a grid of numbers. **Channels** are how many numbers describe each pixel.

| | Channels | Each pixel is | Tensor shape for one 224×224 image |
|---|---|---|---|
| **Grayscale** | 1 | one brightness value, 0 (black) to 255 (white) | `(1, 224, 224)` |
| **RGB** | 3 | three values — red, green, blue intensity | `(3, 224, 224)` |

PyTorch orders image tensors as `(channels, height, width)`, written **CHW**. Note this differs
from PIL and OpenCV, which use `(height, width, channels)` — HWC. Mixing the two up is one of
the most common bugs in a new image pipeline, and it usually shows up as a dimension error
several steps later.

### What we have

Notebook 01 found all 852 Dataset A images are **RGB, 3 channels**. They are phone photographs
of pencil on paper, so there is real colour information — paper tint, shadow, pencil darkness —
even though the handwriting itself is essentially grey on off-white.

Dataset B is **grayscale, 1 channel**. We will deal with that when we reach it.

### Why ImageNet models need 3 channels

ResNet and EfficientNet were pretrained on ImageNet, which is full-colour photographs. The very
first convolution layer has weights shaped for **3 input channels**. Feed it a 1-channel tensor
and it fails immediately with a shape mismatch.

The standard fix for grayscale data is to **repeat the single channel three times**, producing
`(3, H, W)` where all three channels are identical. This does not invent colour — it just makes
the tensor the right shape. The pretrained first layer then sees a grey image, which is exactly
what it is.

For Dataset A no conversion is needed; the images are already RGB. The code below handles both
cases so the same pipeline works on Dataset B later.

In [ ]:
# Confirm the channel situation rather than trusting the earlier notebook
sample = splits['S1_clean'].head(200)
modes = [Image.open(p).mode for p in sample['abs_path']]
print('PIL modes in a 200-image sample:', pd.Series(modes).value_counts().to_dict())
print()

# Demonstrate the HWC -> CHW transposition
img = Image.open(sample['abs_path'].iloc[0]).convert('RGB')
arr = np.asarray(img)
print(f'PIL / numpy shape (HWC) : {arr.shape}')
print(f'PyTorch expects  (CHW)  : {(arr.shape[2], arr.shape[0], arr.shape[1])}')
print('\ntorchvision.transforms.ToTensor() does this transposition for us,')
print('and also rescales pixel values from 0-255 integers to 0.0-1.0 floats.')

---
# STAGE 3 — Normalisation

### Why not just use 0–1 pixel values

`ToTensor()` already rescales pixels to the range 0.0–1.0. Normalisation goes one step further
and shifts each channel to roughly **zero mean and unit standard deviation**:

```
normalised = (pixel - mean) / std
```

Networks train faster and more stably when inputs are centred around zero, because the
gradients flowing back through the first layers stay in a sensible range instead of being
systematically biased in one direction.

### Which mean and std

Two valid choices, and they are **not** interchangeable:

**ImageNet statistics** — `mean = [0.485, 0.456, 0.406]`, `std = [0.229, 0.224, 0.225]`.
These are the values ResNet and EfficientNet were pretrained with. A pretrained model's learned
filters expect inputs distributed this way; feeding it differently-normalised data degrades the
transfer, sometimes badly.

**Dataset statistics** — computed from our own training images. Better suited to our specific
data, but only sensible when training from scratch.

### Our decision

We use **ImageNet statistics everywhere**, including for the simple CNN.

The simple CNN would train slightly better with our own statistics, but using one scheme
throughout means the CNN and ResNet results differ only by architecture. Comparability is worth
more to us than a small optimisation on one model.

We still compute our dataset's own statistics below — partly because it is a useful thing to
know, and partly because it tells us how far our images sit from ImageNet's distribution.

In [ ]:
def compute_dataset_stats(paths, max_images=400, size=IMG_SIZE):
    """Compute per-channel mean and std over a sample of images, in 0-1 scale."""
    sums     = np.zeros(3)
    sums_sq  = np.zeros(3)
    n_pixels = 0

    for p in paths[:max_images]:
        arr = np.asarray(Image.open(p).convert('RGB').resize((size, size))) / 255.0
        sums     += arr.sum(axis=(0, 1))
        sums_sq  += (arr ** 2).sum(axis=(0, 1))
        n_pixels += arr.shape[0] * arr.shape[1]

    mean = sums / n_pixels
    std  = np.sqrt(sums_sq / n_pixels - mean ** 2)
    return mean, std


# IMPORTANT: statistics are computed on TRAINING images only.
# Using val/test images here would leak information about the test set into
# the preprocessing, which is a subtle but real form of test-set contamination.
train_paths = list(splits['S1_clean'].query('split == "train"')['abs_path'])
ds_mean, ds_std = compute_dataset_stats(train_paths)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print('Dataset A (train split) statistics:')
print(f'  mean : {np.round(ds_mean, 4).tolist()}')
print(f'  std  : {np.round(ds_std, 4).tolist()}')
print('\nImageNet statistics (what we will actually use):')
print(f'  mean : {IMAGENET_MEAN}')
print(f'  std  : {IMAGENET_STD}')
print('\nOur images are noticeably brighter and lower-contrast than ImageNet —')
print('unsurprising for photographs of pencil on pale paper.')

---
# STAGE 4 — The Transform Pipelines

### Two pipelines, not one

**Training transform** and **evaluation transform** must be defined separately, because
augmentation (random flips, rotations, colour jitter) belongs in training only.

If augmentation ran at evaluation time, the same image would score differently on each run and
the test number would be meaningless.

### No augmentation yet

Per our experiment plan, the baseline runs with **no augmentation at all**. We add it later as
a controlled comparison — one change at a time, so we can attribute any difference to that
change. Adding augmentation now would leave us unable to say whether an improvement came from
the augmentation or from something else.

So for now both pipelines do the same three things: pad to square, resize to 224, normalise.

In [ ]:
import torchvision.transforms as T


class PadToSquare:
    """Pad the shorter side so the image becomes square, WITHOUT stretching it.

    The fill colour is sampled from the image's own four corners, which on these
    tight character crops are almost always blank page. A single global fill
    value would still show as a band, because every photograph has its own
    exposure — and if brightness happens to differ between classes or sessions,
    that band becomes a cue the network can learn from instead of handwriting.

    Applied BEFORE resizing, so the resize never distorts the aspect ratio.
    """

    def __init__(self, fallback_fill: int = PAD_VALUE, corner: int = 8):
        self.fallback_fill = fallback_fill
        self.corner = corner

    def _sample_paper_colour(self, img: Image.Image):
        """Median colour of the four corner patches — robust to a stray stroke."""
        arr = np.asarray(img)
        c = self.corner
        if arr.ndim != 3 or min(arr.shape[:2]) < 2 * c:
            return (self.fallback_fill,) * len(img.getbands())

        patches = np.concatenate([
            arr[:c, :c].reshape(-1, arr.shape[2]),
            arr[:c, -c:].reshape(-1, arr.shape[2]),
            arr[-c:, :c].reshape(-1, arr.shape[2]),
            arr[-c:, -c:].reshape(-1, arr.shape[2]),
        ])
        return tuple(int(v) for v in np.median(patches, axis=0))

    def __call__(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        if w == h:
            return img

        side = max(w, h)
        left = (side - w) // 2        # centre the original on the new canvas
        top  = (side - h) // 2

        canvas = Image.new(img.mode, (side, side), self._sample_paper_colour(img))
        canvas.paste(img, (left, top))
        return canvas

    def __repr__(self):
        return f'{self.__class__.__name__}(per_image_corner_median, fallback={self.fallback_fill})'


def build_transforms(img_size: int = IMG_SIZE,
                     mean=IMAGENET_MEAN, std=IMAGENET_STD,
                     augment: bool = False):
    """Return (train_transform, eval_transform).

    augment=False for all baseline experiments; augmentation is introduced
    later as a separate, controlled comparison.
    """
    base = [
        T.Lambda(lambda im: im.convert('RGB')),   # also handles 1-channel input
        PadToSquare(PAD_VALUE),
        T.Resize((img_size, img_size)),
    ]
    finish = [
        T.ToTensor(),                              # HWC uint8 -> CHW float in [0,1]
        T.Normalize(mean=mean, std=std),
    ]

    train_tf = T.Compose(base + finish)            # identical to eval for now
    eval_tf  = T.Compose(base + finish)
    return train_tf, eval_tf


train_tf, eval_tf = build_transforms()
print(train_tf)

---
# STAGE 5 — The PyTorch `Dataset` Class

### What a `Dataset` is

A PyTorch `Dataset` is an object that answers two questions:

1. `__len__` — how many samples are there?
2. `__getitem__(i)` — give me sample number `i`, preprocessed and ready

That is the whole contract. Everything else — batching, shuffling, parallel loading — is the
`DataLoader`'s job.

### Why we write our own instead of using `ImageFolder`

`torchvision.datasets.ImageFolder` reads images from class-named folders. It would work here,
and it would be **wrong**: it reads whatever is on disk, so it would silently re-include the
214 duplicates and 20 contradictory files we removed in Notebook 02.

Our `Dataset` reads from a **manifest DataFrame**. The cleaning decisions are therefore
enforced every time data is loaded, and the exact file list is inspectable as a CSV.

### Label encoding

Models need integers, not the strings `'Yes'` / `'No'`. We fix the mapping explicitly:

```
No  -> 0    (negative class)
Yes -> 1    (positive class)
```

`Yes = 1` because it is the class of interest for sensitivity and recall later. Keep this
mapping consistent everywhere — a silently flipped mapping turns a good model into a terrible
one and is painful to spot.

**Reminder:** `Yes` is the label the dataset supplied. It is not a diagnosis.

In [ ]:
from torch.utils.data import Dataset, DataLoader

CLASS_TO_IDX = {'No': 0, 'Yes': 1}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}


class HandwritingDataset(Dataset):
    """Loads handwriting images listed in a split manifest.

    Args:
        manifest : DataFrame with columns abs_path, class_label, sample_id
        split    : which split to use ('train' / 'val' / 'test'), or None for all
        transform: torchvision transform applied to each image
    """

    def __init__(self, manifest: pd.DataFrame, split: str = None, transform=None):
        df = manifest if split is None else manifest[manifest['split'] == split]
        self.df = df.reset_index(drop=True)
        self.transform = transform

        unknown = set(self.df['class_label']) - set(CLASS_TO_IDX)
        assert not unknown, f'Unexpected class labels: {unknown}'

        self.labels = self.df['class_label'].map(CLASS_TO_IDX).values

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img = Image.open(row['abs_path'])

        if self.transform is not None:
            img = self.transform(img)

        label = int(self.labels[idx])
        return img, label

    def class_counts(self) -> dict:
        return self.df['class_label'].value_counts().to_dict()

    def __repr__(self):
        return (f'HandwritingDataset(n={len(self)}, '
                f'counts={self.class_counts()})')


# Quick check on the cleaned split
train_ds = HandwritingDataset(splits['S1_clean'], 'train', train_tf)
val_ds   = HandwritingDataset(splits['S1_clean'], 'val',   eval_tf)
test_ds  = HandwritingDataset(splits['S1_clean'], 'test',  eval_tf)

print('train :', train_ds)
print('val   :', val_ds)
print('test  :', test_ds)

---
# STAGE 6 — The `DataLoader`

### What it adds

The `Dataset` gives one sample at a time. The `DataLoader` wraps it and handles:

- **Batching** — stacks N samples into one tensor of shape `(N, 3, 224, 224)`. Networks process
  batches far more efficiently than single images, and the batch also stabilises the gradient.
- **Shuffling** — reorders the training data every epoch, so the model never sees the same
  sequence twice and cannot learn anything from the ordering.
- **Parallel loading** — `num_workers` background processes decode the next batch while the GPU
  is busy with the current one.

### The settings we use and why

| Setting | Value | Reason |
|---|---|---|
| `batch_size` | 32 | Standard starting point; fits comfortably in a T4's memory at 224×224 |
| `shuffle` | `True` for train, `False` for val/test | Shuffling evaluation data changes nothing except making results harder to compare |
| `num_workers` | 2 | Colab gives two CPU cores; more workers would fight each other |
| `pin_memory` | `True` on GPU | Speeds up the CPU→GPU transfer |
| `drop_last` | `False` | With only ~430 training images we cannot afford to discard a partial batch |

In [ ]:
BATCH_SIZE  = 32
NUM_WORKERS = 2


def build_loaders(manifest: pd.DataFrame,
                  train_transform, eval_transform,
                  batch_size: int = BATCH_SIZE,
                  num_workers: int = NUM_WORKERS,
                  seed: int = SEED):
    """Build train/val/test DataLoaders from one split manifest."""
    datasets = {
        'train': HandwritingDataset(manifest, 'train', train_transform),
        'val'  : HandwritingDataset(manifest, 'val',   eval_transform),
        'test' : HandwritingDataset(manifest, 'test',  eval_transform),
    }

    generator = torch.Generator()
    generator.manual_seed(seed)          # reproducible shuffling

    loaders = {}
    for name, ds in datasets.items():
        loaders[name] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=(name == 'train'),
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
            generator=generator if name == 'train' else None,
        )
    return datasets, loaders


datasets, loaders = build_loaders(splits['S1_clean'], train_tf, eval_tf)

for name, dl in loaders.items():
    print(f'{name:<6} {len(dl.dataset):>4} images  ->  {len(dl):>2} batches of up to {BATCH_SIZE}')

---
# STAGE 7 — Debugging Workflow

Our project rule: **never start a long training run before proving the pipeline works on a few
images.** The checks below take seconds and catch the failures that otherwise waste half an hour.

### 7.1 — Load a handful of samples and print shapes

In [ ]:
print('--- single sample ---')
img, label = train_ds[0]
print(f'type          : {type(img)}')
print(f'shape         : {tuple(img.shape)}   (channels, height, width)')
print(f'dtype         : {img.dtype}')
print(f'value range   : {img.min():.3f} to {img.max():.3f}')
print(f'label         : {label}  ->  {IDX_TO_CLASS[label]}')
print()

print('--- first 10 samples ---')
for i in range(10):
    img, label = train_ds[i]
    row = train_ds.df.iloc[i]
    print(f'  [{i}] {tuple(img.shape)}  label={label} ({IDX_TO_CLASS[label]:<3})  '
          f'orig={row["width"]}x{row["height"]}  {row["sample_id"]}')

**What to check in that output.**

- Every shape must be `(3, 224, 224)` — identical regardless of the original dimensions listed
  on the right. That proves padding and resizing work across all 538 original sizes.
- The value range should be roughly **−2 to +2**, not 0 to 1. After normalisation, values are
  standard deviations from the mean, so negative numbers are expected and correct.
- Labels must be integers 0 or 1, and must match the class name shown beside them.

### 7.2 — Pull one batch through the loader

In [ ]:
images, labels = next(iter(loaders['train']))

print(f'batch images shape : {tuple(images.shape)}')
print(f'batch labels shape : {tuple(labels.shape)}')
print(f'images dtype       : {images.dtype}')
print(f'labels dtype       : {labels.dtype}')
print(f'labels in batch    : {labels.tolist()}')
print(f'class balance      : '
      f'No={int((labels == 0).sum())}, Yes={int((labels == 1).sum())}')
print()

# Move to the device the model will live on
images_gpu = images.to(DEVICE)
print(f'after .to(DEVICE)  : {images_gpu.device}, shape {tuple(images_gpu.shape)}')

# Approximate memory footprint of one batch
mb = images.element_size() * images.nelement() / 1024**2
print(f'one batch occupies : {mb:.1f} MB')

### 7.3 — Look at the tensors

This is the check that catches what shape assertions cannot.

We **denormalise** the tensors back to viewable images and plot them. If the padding colour is
wrong, the channels are transposed, or the normalisation is inverted, it is obvious here and
invisible everywhere else.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 110


def denormalise(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Undo Normalize() so a tensor can be displayed. Inverse of (x - mean) / std."""
    mean = torch.tensor(mean).view(3, 1, 1)
    std  = torch.tensor(std).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1).permute(1, 2, 0)   # CHW -> HWC


fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax, img, lab in zip(axes.ravel(), images, labels):
    ax.imshow(denormalise(img))
    ax.set_title(f'{IDX_TO_CLASS[int(lab)]}', fontsize=9)
    ax.axis('off')

fig.suptitle('One training batch after preprocessing (denormalised for display)', fontsize=12)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'A_preprocessed_batch.png', bbox_inches='tight')
plt.show()
print(f'[saved] {FIGURES_DIR / "A_preprocessed_batch.png"}')

**What to check.** Every image should be square, the handwriting should be undistorted and
centred, and the padded margins should blend with the paper rather than showing as dark bands.
If a character looks stretched, `PadToSquare` is not running.

### 7.4 — Before and after, side by side

The clearest evidence that aspect ratio is preserved: take the most extreme non-square images
in the dataset and compare original against preprocessed.

In [ ]:
# Pick the images with the most extreme aspect ratios
d = splits['S1_clean'].copy()
d['aspect'] = d['width'] / d['height']
extreme = pd.concat([d.nsmallest(3, 'aspect'), d.nlargest(3, 'aspect')])

fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for col, (_, row) in enumerate(extreme.iterrows()):
    original = Image.open(row['abs_path']).convert('RGB')
    axes[0, col].imshow(original)
    axes[0, col].set_title(f'original\n{row["width"]}x{row["height"]}  '
                           f'ar={row["aspect"]:.2f}', fontsize=8)
    axes[0, col].axis('off')

    processed = eval_tf(Image.open(row['abs_path']))
    axes[1, col].imshow(denormalise(processed))
    axes[1, col].set_title(f'preprocessed\n224x224', fontsize=8)
    axes[1, col].axis('off')

fig.suptitle('Aspect ratio preservation: the six most extreme images in A2_clean', fontsize=12)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'A_aspect_preservation.png', bbox_inches='tight')
plt.show()
print(f'[saved] {FIGURES_DIR / "A_aspect_preservation.png"}')

### 7.5 — Timing check

How long does one pass over the training data take, just to load it? If this is slow, training
will be dominated by data loading rather than by the GPU, and we would raise `num_workers`.

In [ ]:
t0 = time.time()
n_images = 0
for images_b, labels_b in loaders['train']:
    n_images += len(labels_b)
elapsed = time.time() - t0

print(f'one full pass over the training set: {n_images} images in {elapsed:.1f}s')
print(f'  {n_images/elapsed:.0f} images/second')
print(f'\nestimated data-loading time for 20 epochs: {20*elapsed/60:.1f} minutes')
print('(actual training will be slower — this measures loading only)')

---
# STAGE 8 — Class Weights (Computed, Not Applied)

A2_clean is imbalanced at roughly **2 Yes : 1 No**. A model that predicted `Yes` for every
single image would score about 66% accuracy while being completely useless.

There are standard remedies — weighted loss, oversampling, focal loss — but our project rule is
to **measure the problem before treating it**. We compute the weights now and keep them in the
config, then decide after seeing the baseline confusion matrix whether they are needed.

**Inverse-frequency weighting** gives each class a weight inversely proportional to how often it
appears, so the rarer class contributes more to the loss:

```
weight_c = n_total / (n_classes × n_samples_in_c)
```

In [ ]:
def compute_class_weights(dataset: HandwritingDataset) -> dict:
    """Inverse-frequency class weights. Computed on TRAINING data only."""
    counts = dataset.df['class_label'].value_counts()
    n_total, n_classes = len(dataset), len(CLASS_TO_IDX)

    weights = {}
    for cls, idx in CLASS_TO_IDX.items():
        n_c = counts.get(cls, 0)
        weights[cls] = round(n_total / (n_classes * n_c), 4) if n_c else 0.0
    return weights


class_weights = compute_class_weights(datasets['train'])

print('training class counts :', datasets['train'].class_counts())
print('inverse-freq weights  :', class_weights)

majority = max(datasets['train'].class_counts().items(), key=lambda kv: kv[1])
print(f'\nMajority-class baseline: always predicting "{majority[0]}" would score '
      f'{100*majority[1]/len(datasets["train"]):.1f}% accuracy on the training set.')
print('Any model we build must beat that, or it has learned nothing.')

---
# STAGE 9 — Save the Preprocessing Config

Every experiment must record exactly how its data was prepared. Without this, a result cannot
be reproduced or fairly compared with another.

In [ ]:
preproc_config = {
    'created'          : datetime.now().isoformat(timespec='seconds'),
    'seed'             : SEED,
    'img_size'         : IMG_SIZE,
    'pad_value'        : PAD_VALUE,
    'pad_value_source' : 'per-image corner median (global value is fallback only)',
    'resize_strategy'  : 'pad_to_square_then_resize',
    'channels'         : 3,
    'normalisation'    : 'imagenet',
    'imagenet_mean'    : IMAGENET_MEAN,
    'imagenet_std'     : IMAGENET_STD,
    'dataset_mean'     : np.round(ds_mean, 4).tolist(),
    'dataset_std'      : np.round(ds_std, 4).tolist(),
    'augmentation'     : 'none',
    'batch_size'       : BATCH_SIZE,
    'num_workers'      : NUM_WORKERS,
    'class_to_idx'     : CLASS_TO_IDX,
    'class_weights'    : class_weights,
    'torch_version'    : torch.__version__,
    'torchvision_version': torchvision.__version__,
    'device'           : str(DEVICE),
}

CONFIG_PATH = EXPERIMENTS_DIR / 'preprocessing_config.json'
with open(CONFIG_PATH, 'w') as f:
    json.dump(preproc_config, f, indent=2)

print(json.dumps(preproc_config, indent=2))
print(f'\n[saved] {CONFIG_PATH}')

---
# Experiment Log for the Supervisor

## What we actually did

1. Established that all Dataset A images are RGB 3-channel, confirming it rather than assuming.
2. Chose **pad-to-square then resize** over direct resizing or cropping, because handwriting
   shape is the signal and stretching or cropping it destroys information.
3. Measured the paper colour from image corners and padded with a matching light value, so the
   padding does not create an artificial border the model could learn from.
4. Fixed the output size at 224×224 so the simple CNN and the pretrained models all receive
   identically preprocessed data.
5. Computed our dataset's own channel statistics from **training images only**, then chose
   ImageNet statistics for consistency with the pretrained models we will use.
6. Wrote a `Dataset` class that reads from split manifests rather than folders, so the cleaning
   decisions from Notebook 02 are enforced at load time.
7. Verified the pipeline: tensor shapes, value ranges, batch shapes, device transfer, and a
   visual check that preprocessed tensors still contain undistorted handwriting.
8. Computed the majority-class baseline that any model must beat.

## What we learned

- The pad-then-resize approach handles all 538 original image sizes into one uniform shape
  without distorting any of them — confirmed visually on the six most extreme aspect ratios.
- Our images are brighter and lower-contrast than ImageNet, which is expected for pencil on
  pale paper and worth noting when we interpret transfer-learning results.
- The imbalance means **accuracy alone will be misleading**. We will need per-class metrics from
  the first experiment onward.

## What we can tell the supervisor

> We built the data pipeline to read from the audited manifests rather than from the image
> folders, so the duplicates and contradictory labels we removed cannot re-enter an experiment
> by accident. Preprocessing preserves handwriting shape rather than stretching it to fit, and
> we verified visually that the tensors reaching the model still contain undistorted characters.
> We also established the majority-class baseline the model has to beat before any accuracy
> figure means anything.

## Limitations carried forward

| Limitation | Effect |
|---|---|
| ~2:1 class imbalance | Accuracy is not a sufficient metric; needs precision/recall/specificity |
| Padding adds blank area on extreme aspect ratios | Some images carry less usable signal per pixel |
| ImageNet normalisation is not tuned to our data | Slightly suboptimal for the from-scratch CNN; accepted for comparability |
| Small training set (~430 images) | Overfitting is likely; we will watch for it from epoch 1 |

---

# Next notebook

**Notebook 04 — Baseline CNN and the training loop**

- What each layer does: convolution, ReLU, pooling, flatten, fully connected
- Input and output shapes at every stage
- Loss function and optimizer, explained before they are used
- The remaining debugging steps: one batch through the model, one loss, one optimizer step
- Then the real training run, with history tracking and curves

Run this notebook and send me the Stage 7.1 and 7.2 output.